<a href="https://colab.research.google.com/github/STAKTIME/krutoiAIraspoznovatelBot/blob/main/%D0%91%D0%BE%D1%82%D1%8C_%D0%BC%D0%B5%D1%82%D0%B0%D0%BB%D0%BB%D1%8B_%D1%80%D0%B0%D1%81%D0%BF%D0%BE%D0%B7%D0%BD%D0%BE%D1%8E%D1%89%D1%8A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pytelegrambotapi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.4/334.4 kB 8.3 MB/s eta 0:00:00


In [6]:
import telebot
import time
import threading

API_TOKEN = 'А знать вот надо'
bot = telebot.TeleBot(API_TOKEN)

@bot.message_handler(content_types=['photo'])
def burning_libr(message):
    # Скачиваем фото
    file_info = bot.get_file(message.photo[-1].file_id)
    file_name = file_info.file_path.split('/')[-1]
    downloaded_file = bot.download_file(file_info.file_path)
    with open(file_name, 'wb') as new_file:
        new_file.write(downloaded_file)

    # Отправляем стартовое сообщение
    msg = bot.reply_to(message, "Понял, принял, обрабатываю")

    # Контейнер для результата (будет заполнен из потока)
    result_container = {"value": None, "error": None}

    # Функция, которая запустится в фоне
    def background_task():
        try:
            res = recognizer(file_name)  # ваша тяжёлая функция
            result_container["value"] = res
        except Exception as e:
            result_container["error"] = str(e)

    # Запускаем поток
    thread = threading.Thread(target=background_task)
    thread.start()

    # Анимация загрузки: пока поток жив, обновляем сообщение с точками
    dots = 0
    while thread.is_alive():
        # Формируем текст с нужным количеством точек (0, 1, 2, 3)
        text = "Понял, принял, обрабатываю" + "." * dots
        try:
            bot.edit_message_text(text, chat_id=message.chat.id, message_id=msg.message_id)
        except Exception:
            # Если сообщение не удалось отредактировать (например, уже удалено), игнорируем
            pass

        dots = (dots + 1) % 4  # цикл 0→1→2→3→0...
        time.sleep(0.5)

    # Поток завершён – проверяем результат
    if result_container["error"]:
        bot.send_message(message.chat.id, f"Ошибка: {result_container['error']}")
    else:
        bot.send_message(message.chat.id, f"Результат: {result_container['value']}")

bot.polling()

1/1 [==============================] - 1s 1s/step


In [4]:
!unzip ./converted_keras.zip

Archive:  ./converted_keras.zip
 extracting: keras_model.h5          
 extracting: labels.txt              


In [ ]:
!pip install -q tf-keras==2.19.0 h5py==3.11.0

In [5]:
import tf_keras as keras
from tf_keras.models import load_model
from PIL import Image, ImageOps
import numpy as np

def recognizer(image_path):
  np.set_printoptions(suppress=True)
  model = load_model("keras_model.h5", compile=False)
  class_names = open("labels.txt", "r").readlines()
  data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
  image = Image.open(image_path).convert("RGB")
  size = (224, 224)
  image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)
  image_array = np.asarray(image)
  normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
  data[0] = normalized_image_array
  prediction = model.predict(data)
  index = np.argmax(prediction)
  class_name = class_names[index]

  return class_name[2:]


In [ ]:
recognizer('/content/тест/carbon-1.png')

1/1 [==============================] - 2s 2s/step


'Медь\n'